# Journal Extension J1 — Original Protocol Lock & Reproducibility

This notebook freezes the **original thesis internal protocol** that produced the main result.

### Canonical protocol
- ISIC 2019, 8 classes
- stratified **80/10/10**
- seed **42**
- TRAIN **20,264**
- Validation **2,533**
- Test **2,534**
- final validation-selected method: `NoTTA_StackingBlend_ClassBias`
- internal test BACC: about **83.95%**

### Required Kaggle inputs

Attach **two saved notebook versions WITH OUTPUTS**:

1. **Original full thesis pipeline** (`last-part-of-the-previous-one`)
   - must contain:
   - `rotation_ssl_paperlike/splits/rotation_fixed_train_val_test_split.json`

2. **Confidence-gated final ensemble** (`confidence-gated-ensemble`)
   - must contain:
   - `reuse_only_all_model_ensemble/final_ensemble_config.json`
   - `reuse_only_all_model_ensemble/final_internal_probabilities.npz`
   - `reuse_only_all_model_ensemble/internal_test_all_models_comparison.csv`
   - `reuse_only_all_model_ensemble/validation_final_ensemble_candidates.csv`

### Not required for J1
No raw ISIC images, official ISIC test images, PAD-UFES-20, grouped Stage outputs, or DINO outputs are needed.

### Main output
`J1_ORIGINAL_PROTOCOL_LOCK_PACKAGE.zip`

All later journal notebooks should use the locked split produced here.

In [1]:
# CELL 1 — SETUP
import json
import hashlib
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize

KAGGLE_INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

OUTPUT_DIR = WORKING / "j1_original_protocol_lock"
TABLE_DIR = OUTPUT_DIR / "tables"
AUDIT_DIR = OUTPUT_DIR / "audit"
SOURCE_DIR = OUTPUT_DIR / "source_artifacts"

for d in [OUTPUT_DIR, TABLE_DIR, AUDIT_DIR, SOURCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED_EXPECTED = 42
TRAIN_EXPECTED = 20264
VAL_EXPECTED = 2533
TEST_EXPECTED = 2534
NUM_CLASSES = 8

CLASS_NAMES = ["MEL","NV","BCC","AK","BKL","DF","VASC","SCC"]
FINAL_METHOD_EXPECTED = "NoTTA_StackingBlend_ClassBias"
FINAL_BACC_REFERENCE = 0.839549

print("=" * 92)
print("J1 — ORIGINAL THESIS PROTOCOL LOCK")
print("=" * 92)
print("Output directory:", OUTPUT_DIR)

J1 — ORIGINAL THESIS PROTOCOL LOCK
Output directory: /kaggle/working/j1_original_protocol_lock


In [2]:
# CELL 2 — FIND THE CANONICAL SAVED OUTPUT FILES

def find_all(filename):
    return sorted(
        KAGGLE_INPUT.rglob(filename),
        key=lambda p: (len(p.parts), str(p))
    )

def select_split_file():
    hits = find_all("rotation_fixed_train_val_test_split.json")
    preferred = [
        p for p in hits
        if "rotation_ssl_paperlike" in str(p)
        and "splits" in str(p)
    ]
    candidates = preferred if preferred else hits
    if not candidates:
        raise FileNotFoundError(
            "rotation_fixed_train_val_test_split.json was not found. "
            "Attach the saved output of `last-part-of-the-previous-one`."
        )
    return candidates[0], hits

def select_confidence_file(filename):
    hits = find_all(filename)
    preferred = [
        p for p in hits
        if "reuse_only_all_model_ensemble" in str(p)
    ]
    candidates = preferred if preferred else hits
    if not candidates:
        raise FileNotFoundError(
            f"{filename} was not found. "
            "Attach the saved output of `confidence-gated-ensemble`."
        )
    return candidates[0], hits

SPLIT_JSON, SPLIT_HITS = select_split_file()
FINAL_CONFIG_JSON, CONFIG_HITS = select_confidence_file("final_ensemble_config.json")
FINAL_PROB_NPZ, PROB_HITS = select_confidence_file("final_internal_probabilities.npz")
COMPARISON_CSV, COMPARISON_HITS = select_confidence_file("internal_test_all_models_comparison.csv")
VALIDATION_CANDIDATES_CSV, CANDIDATE_HITS = select_confidence_file("validation_final_ensemble_candidates.csv")

print("Original split:")
print(SPLIT_JSON)
print("\nFinal ensemble config:")
print(FINAL_CONFIG_JSON)
print("\nFinal probabilities:")
print(FINAL_PROB_NPZ)
print("\nInternal comparison:")
print(COMPARISON_CSV)
print("\nValidation candidates:")
print(VALIDATION_CANDIDATES_CSV)

discovery = {
    "selected": {
        "split_json": str(SPLIT_JSON),
        "final_config_json": str(FINAL_CONFIG_JSON),
        "final_probabilities": str(FINAL_PROB_NPZ),
        "comparison_csv": str(COMPARISON_CSV),
        "validation_candidates_csv": str(VALIDATION_CANDIDATES_CSV),
    },
    "all_matches": {
        "split_json": [str(p) for p in SPLIT_HITS],
        "final_config_json": [str(p) for p in CONFIG_HITS],
        "final_probabilities": [str(p) for p in PROB_HITS],
        "comparison_csv": [str(p) for p in COMPARISON_HITS],
        "validation_candidates_csv": [str(p) for p in CANDIDATE_HITS],
    },
}

with open(AUDIT_DIR / "input_discovery.json", "w") as f:
    json.dump(discovery, f, indent=2)

Original split:
/kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/splits/rotation_fixed_train_val_test_split.json

Final ensemble config:
/kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/reuse_only_all_model_ensemble/final_ensemble_config.json

Final probabilities:
/kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/reuse_only_all_model_ensemble/final_internal_probabilities.npz

Internal comparison:
/kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/reuse_only_all_model_ensemble/internal_test_all_models_comparison.csv

Validation candidates:
/kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/reuse_only_all_model_ensemble/validation_final_ensemble_candidates.csv


In [3]:
# CELL 3 — LOAD AND VERIFY THE EXACT ORIGINAL 80/10/10 SPLIT

with open(SPLIT_JSON, "r") as f:
    split_info = json.load(f)

TRAIN_DF = pd.DataFrame(split_info["train"])
VAL_DF = pd.DataFrame(split_info["val"])
TEST_DF = pd.DataFrame(split_info["test"])

for name, df in [("TRAIN", TRAIN_DF), ("VAL", VAL_DF), ("TEST", TEST_DF)]:
    if not {"filepath", "label"}.issubset(df.columns):
        raise ValueError(f"{name} does not contain filepath + label.")
    df["filepath"] = df["filepath"].astype(str)
    df["label"] = df["label"].astype(int)
    df["image_id"] = df["filepath"].map(lambda x: Path(x).stem)
    df["class_name"] = df["label"].map({i: CLASS_NAMES[i] for i in range(NUM_CLASSES)})

checks = [
    ("seed", int(split_info["seed"]) == 42, split_info["seed"], 42),
    ("train_ratio", np.isclose(split_info["train_ratio"], 0.80), split_info["train_ratio"], 0.80),
    ("val_ratio", np.isclose(split_info["val_ratio"], 0.10), split_info["val_ratio"], 0.10),
    ("test_ratio", np.isclose(split_info["test_ratio"], 0.10), split_info["test_ratio"], 0.10),
    ("num_classes", int(split_info["num_classes"]) == 8, split_info["num_classes"], 8),
    ("train_size", len(TRAIN_DF) == TRAIN_EXPECTED, len(TRAIN_DF), TRAIN_EXPECTED),
    ("val_size", len(VAL_DF) == VAL_EXPECTED, len(VAL_DF), VAL_EXPECTED),
    ("test_size", len(TEST_DF) == TEST_EXPECTED, len(TEST_DF), TEST_EXPECTED),
]

CHECK_DF = pd.DataFrame(
    [{"check": n, "passed": bool(p), "observed": o, "expected": e}
     for n, p, o, e in checks]
)

display(CHECK_DF)
CHECK_DF.to_csv(AUDIT_DIR / "protocol_assertions.csv", index=False)

if not CHECK_DF["passed"].all():
    raise RuntimeError(
        "Attached split is not the expected original thesis split.\n"
        + CHECK_DF.loc[~CHECK_DF["passed"]].to_string(index=False)
    )

print("\nExact protocol sizes:")
print("TRAIN:", len(TRAIN_DF))
print("VAL  :", len(VAL_DF))
print("TEST :", len(TEST_DF))
print("\nProtocol assertions: PASSED")

         check  passed  observed  expected
0         seed    True      42.0      42.0
1  train_ratio    True       0.8       0.8
2    val_ratio    True       0.1       0.1
3   test_ratio    True       0.1       0.1
4  num_classes    True       8.0       8.0
5   train_size    True   20264.0   20264.0
6     val_size    True    2533.0    2533.0
7    test_size    True    2534.0    2534.0


Exact protocol sizes:
TRAIN: 20264
VAL  : 2533
TEST : 2534

Protocol assertions: PASSED


In [4]:
# CELL 4 — ID OVERLAP, DUPLICATES, CLASS DISTRIBUTION

sets = {
    "TRAIN": set(TRAIN_DF["image_id"]),
    "VAL": set(VAL_DF["image_id"]),
    "TEST": set(TEST_DF["image_id"]),
}

OVERLAP_DF = pd.DataFrame([
    {"comparison": "TRAIN_vs_VAL", "overlap_count": len(sets["TRAIN"] & sets["VAL"])},
    {"comparison": "TRAIN_vs_TEST", "overlap_count": len(sets["TRAIN"] & sets["TEST"])},
    {"comparison": "VAL_vs_TEST", "overlap_count": len(sets["VAL"] & sets["TEST"])},
])

DUPLICATE_DF = pd.DataFrame([
    {
        "split": name,
        "duplicate_image_ids": int(df["image_id"].duplicated().sum()),
        "duplicate_filepaths": int(df["filepath"].duplicated().sum()),
    }
    for name, df in [("TRAIN", TRAIN_DF), ("VAL", VAL_DF), ("TEST", TEST_DF)]
])

display(OVERLAP_DF)
display(DUPLICATE_DF)

OVERLAP_DF.to_csv(AUDIT_DIR / "split_id_overlap_audit.csv", index=False)
DUPLICATE_DF.to_csv(AUDIT_DIR / "within_split_duplicate_audit.csv", index=False)

if OVERLAP_DF["overlap_count"].sum() != 0:
    raise RuntimeError("Exact image IDs overlap across splits.")

if DUPLICATE_DF[["duplicate_image_ids", "duplicate_filepaths"]].to_numpy().sum() != 0:
    raise RuntimeError("Duplicate exact image IDs/filepaths exist inside a split.")

def make_portable(df, split_name):
    out = df[["image_id", "label", "class_name", "filepath"]].copy()
    out.insert(0, "split", split_name)
    return out.rename(columns={"filepath": "original_filepath"})

LOCKED_SPLIT_DF = pd.concat([
    make_portable(TRAIN_DF, "TRAIN"),
    make_portable(VAL_DF, "VAL"),
    make_portable(TEST_DF, "TEST"),
], ignore_index=True)

LOCKED_IDS_DF = LOCKED_SPLIT_DF[
    ["split", "image_id", "label", "class_name"]
].copy()

LOCKED_SPLIT_DF.to_csv(
    TABLE_DIR / "original_80_10_10_locked_split_with_provenance.csv",
    index=False,
)
LOCKED_IDS_DF.to_csv(
    TABLE_DIR / "original_80_10_10_locked_ids.csv",
    index=False,
)

count_table = (
    LOCKED_IDS_DF
    .groupby(["split", "class_name"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=["TRAIN", "VAL", "TEST"], columns=CLASS_NAMES)
)

percent_table = count_table.div(count_table.sum(axis=1), axis=0) * 100.0

display(count_table)

count_table.to_csv(TABLE_DIR / "class_distribution_counts.csv")
percent_table.to_csv(TABLE_DIR / "class_distribution_percent.csv")

print("\nExact ID overlap/duplicate audit: PASSED")

      comparison  overlap_count
0   TRAIN_vs_VAL              0
1  TRAIN_vs_TEST              0
2    VAL_vs_TEST              0

   split  duplicate_image_ids  duplicate_filepaths
0  TRAIN                    0                    0
1    VAL                    0                    0
2   TEST                    0                    0

class_name   MEL     NV   BCC   AK   BKL   DF  VASC  SCC
split                                                   
TRAIN       3618  10300  2658  694  2099  191   202  502
VAL          452   1287   332   87   262   24    26   63
TEST         452   1288   333   86   263   24    25   63


Exact ID overlap/duplicate audit: PASSED


In [5]:
# CELL 5 — CREATE SHA-256 PROTOCOL HASHES

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def ordered_id_label_hash(df):
    text = "\n".join(
        f"{row.image_id}|{int(row.label)}"
        for row in df.itertuples(index=False)
    )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

TRAIN_HASH = ordered_id_label_hash(TRAIN_DF)
VAL_HASH = ordered_id_label_hash(VAL_DF)
TEST_HASH = ordered_id_label_hash(TEST_DF)

combined_text = (
    f"seed=42\n"
    f"ratios=0.80,0.10,0.10\n"
    f"train={TRAIN_HASH}\n"
    f"val={VAL_HASH}\n"
    f"test={TEST_HASH}\n"
)

COMBINED_PROTOCOL_HASH = hashlib.sha256(
    combined_text.encode("utf-8")
).hexdigest()

PROTOCOL_HASHES = {
    "train_ordered_id_label_sha256": TRAIN_HASH,
    "validation_ordered_id_label_sha256": VAL_HASH,
    "test_ordered_id_label_sha256": TEST_HASH,
    "combined_protocol_sha256": COMBINED_PROTOCOL_HASH,
    "source_split_json_sha256": sha256_file(SPLIT_JSON),
    "source_final_config_sha256": sha256_file(FINAL_CONFIG_JSON),
    "source_final_probabilities_sha256": sha256_file(FINAL_PROB_NPZ),
}

with open(AUDIT_DIR / "protocol_hashes.json", "w") as f:
    json.dump(PROTOCOL_HASHES, f, indent=2)

print(json.dumps(PROTOCOL_HASHES, indent=2))

{
  "train_ordered_id_label_sha256": "9fab7774e76d561c72da185315f2376b46d351f36c8197af69ea829ea61d278d",
  "validation_ordered_id_label_sha256": "4afa813a910a84b6346cd4421476fc307c225eb3c25736508d9ae2997ca3e8e5",
  "test_ordered_id_label_sha256": "2c9d9c9dc871c4525e0b71dab54f684ed3a96cc3b4fb0d779964cba6a672f312",
  "combined_protocol_sha256": "b483ced043583491ed0793a5e80ecd1192055230393b6603200a3441d04f5766",
  "source_split_json_sha256": "bb0335ef5e845360bdc7f225a12fe9a8fee8c83bad22efd91a751b333a5d49e3",
  "source_final_config_sha256": "7c2e5c5d092bb585de429049b6703c9240dba8bf24d56c7e19b9f9feeb53cfe9",
  "source_final_probabilities_sha256": "b3ae4ddbc5f01e4cda935ac74ef796c4713a4e8815d2e7f105afe8e618e6614c"
}


In [6]:
# CELL 6 — FROZEN FINAL MODEL AUDIT + PROBABILITY ALIGNMENT

with open(FINAL_CONFIG_JSON, "r") as f:
    FINAL_CONFIG = json.load(f)

if FINAL_CONFIG["final_method"] != FINAL_METHOD_EXPECTED:
    raise RuntimeError(
        f"Wrong final method loaded: {FINAL_CONFIG['final_method']} "
        f"(expected {FINAL_METHOD_EXPECTED})"
    )

if bool(FINAL_CONFIG["test_used_for_selection"]):
    raise RuntimeError("Saved config says test data were used for model selection.")

if not bool(FINAL_CONFIG["validation_selection_passed"]):
    raise RuntimeError("Saved validation-controlled final selection did not pass.")

with np.load(FINAL_PROB_NPZ) as z:
    Y_VAL = np.asarray(z["y_val"], dtype=np.int32)
    Y_TEST = np.asarray(z["y_test"], dtype=np.int32)
    FINAL_VAL_PROB = np.asarray(z["final_val_prob"], dtype=np.float64)
    FINAL_TEST_PROB = np.asarray(z["final_test_prob"], dtype=np.float64)
    BIAS_VAL_PROB = np.asarray(z["stacking_bias_val_prob"], dtype=np.float64)
    BIAS_TEST_PROB = np.asarray(z["stacking_bias_test_prob"], dtype=np.float64)

locked_y_val = VAL_DF["label"].to_numpy(np.int32)
locked_y_test = TEST_DF["label"].to_numpy(np.int32)

if not np.array_equal(Y_VAL, locked_y_val):
    raise RuntimeError("Final ensemble VAL labels do not align with locked VAL rows.")

if not np.array_equal(Y_TEST, locked_y_test):
    raise RuntimeError("Final ensemble TEST labels do not align with locked TEST rows.")

if FINAL_VAL_PROB.shape != (VAL_EXPECTED, NUM_CLASSES):
    raise RuntimeError(f"Wrong final VAL probability shape: {FINAL_VAL_PROB.shape}")

if FINAL_TEST_PROB.shape != (TEST_EXPECTED, NUM_CLASSES):
    raise RuntimeError(f"Wrong final TEST probability shape: {FINAL_TEST_PROB.shape}")

if not np.allclose(FINAL_VAL_PROB, BIAS_VAL_PROB, atol=1e-7):
    raise RuntimeError("final_val_prob is not the class-bias final probability.")

if not np.allclose(FINAL_TEST_PROB, BIAS_TEST_PROB, atol=1e-7):
    raise RuntimeError("final_test_prob is not the class-bias final probability.")

print("Final method:", FINAL_CONFIG["final_method"])
print("Validation selection passed:", FINAL_CONFIG["validation_selection_passed"])
print("Test used for selection:", FINAL_CONFIG["test_used_for_selection"])
print("Probability alignment: PASSED")

Final method: NoTTA_StackingBlend_ClassBias
Validation selection passed: True
Test used for selection: False
Probability alignment: PASSED


In [7]:
# CELL 7 — RECOMPUTE THE CANONICAL FINAL METRICS

def normalize_prob(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, 1e-12, None)
    return p / p.sum(axis=1, keepdims=True)

def evaluate(y, p, split_name):
    p = normalize_prob(p)
    pred = np.argmax(p, axis=1)

    y_bin = label_binarize(y, classes=np.arange(NUM_CLASSES))
    try:
        auc = float(
            roc_auc_score(
                y_bin, p, average="macro", multi_class="ovr"
            )
        )
    except Exception:
        auc = np.nan

    cm = confusion_matrix(y, pred, labels=np.arange(NUM_CLASSES))
    specificity = []

    for c in range(NUM_CLASSES):
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = cm.sum() - tp - fn - fp
        specificity.append(tn / max(tn + fp, 1))

    return {
        "method": FINAL_CONFIG["final_method"],
        "split": split_name,
        "n_samples": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "bacc": float(balanced_accuracy_score(y, pred)),
        "macro_precision": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "macro_recall": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "macro_f1": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
        "weighted_f1": float(
            f1_score(y, pred, average="weighted", zero_division=0)
        ),
        "macro_specificity": float(np.mean(specificity)),
        "auc_macro_ovr": auc,
    }

VAL_METRICS = evaluate(Y_VAL, FINAL_VAL_PROB, "Validation")
TEST_METRICS = evaluate(Y_TEST, FINAL_TEST_PROB, "InternalTest")

FINAL_METRICS_DF = pd.DataFrame([VAL_METRICS, TEST_METRICS])
display(FINAL_METRICS_DF)

FINAL_METRICS_DF.to_csv(
    TABLE_DIR / "final_model_recomputed_metrics.csv",
    index=False,
)

print(
    "\nRecomputed canonical internal test BACC:",
    f"{100 * TEST_METRICS['bacc']:.4f}%"
)

if abs(TEST_METRICS["bacc"] - FINAL_BACC_REFERENCE) > 0.001:
    raise RuntimeError(
        "Recomputed BACC does not match the expected ~83.95% result."
    )

print("Canonical ~83.95% result verification: PASSED")

                          method         split  n_samples  accuracy      bacc  \
0  NoTTA_StackingBlend_ClassBias    Validation       2533  0.836557  0.820332   
1  NoTTA_StackingBlend_ClassBias  InternalTest       2534  0.838990  0.839549   

   macro_precision  macro_recall  macro_f1  weighted_f1  macro_specificity  \
0         0.751879      0.820332  0.781634     0.839779           0.973014   
1         0.766652      0.839549  0.797961     0.841509           0.972839   

   auc_macro_ovr  
0       0.975191  
1       0.976656  


Recomputed canonical internal test BACC: 83.9549%
Canonical ~83.95% result verification: PASSED


In [8]:
# CELL 8 — SAVE SELECTION AUDIT, MANIFEST, SOURCE COPIES

VALIDATION_CANDIDATES_DF = pd.read_csv(VALIDATION_CANDIDATES_CSV)
COMPARISON_DF = pd.read_csv(COMPARISON_CSV)

display(VALIDATION_CANDIDATES_DF)
display(COMPARISON_DF)

VALIDATION_CANDIDATES_DF.to_csv(
    TABLE_DIR / "validation_final_ensemble_candidates_locked.csv",
    index=False,
)
COMPARISON_DF.to_csv(
    TABLE_DIR / "internal_test_all_models_comparison_locked.csv",
    index=False,
)

selection_audit = {
    "final_method": FINAL_CONFIG["final_method"],
    "selection_basis": "validation only",
    "validation_selection_passed": bool(
        FINAL_CONFIG["validation_selection_passed"]
    ),
    "test_used_for_selection": bool(
        FINAL_CONFIG["test_used_for_selection"]
    ),
    "saved_validation_selection": FINAL_CONFIG["validation_selection"],
    "recomputed_validation_metrics": VAL_METRICS,
    "recomputed_test_metrics": TEST_METRICS,
}

with open(AUDIT_DIR / "final_model_selection_audit.json", "w") as f:
    json.dump(selection_audit, f, indent=2, default=float)

manifest = {
    "protocol_name": "Original thesis stratified 80/10/10 internal protocol",
    "status": "LOCKED",
    "primary_internal_protocol": True,
    "dataset": "ISIC 2019",
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "seed": 42,
    "split_method": (
        "sklearn train_test_split: first 80/20, then 20% remainder "
        "split 50/50; shuffle=True; stratify=label; random_state=42"
    ),
    "split_sizes": {
        "train": len(TRAIN_DF),
        "validation": len(VAL_DF),
        "test": len(TEST_DF),
    },
    "final_method": FINAL_CONFIG["final_method"],
    "final_internal_test_metrics": TEST_METRICS,
    "test_used_for_selection": False,
    "combined_protocol_sha256": COMBINED_PROTOCOL_HASH,
    "authoritative_split_file": "tables/original_80_10_10_locked_ids.csv",
    "future_rule": (
        "All main internal journal comparisons must use this exact split "
        "and row order. Do not replace it with the grouped robustness split."
    ),
}

with open(OUTPUT_DIR / "J1_ORIGINAL_PROTOCOL_MANIFEST.json", "w") as f:
    json.dump(manifest, f, indent=2, default=float)

source_map = {
    "rotation_fixed_train_val_test_split.json": SPLIT_JSON,
    "final_ensemble_config.json": FINAL_CONFIG_JSON,
    "final_internal_probabilities.npz": FINAL_PROB_NPZ,
    "internal_test_all_models_comparison.csv": COMPARISON_CSV,
    "validation_final_ensemble_candidates.csv": VALIDATION_CANDIDATES_CSV,
}

for name, src in source_map.items():
    shutil.copy2(src, SOURCE_DIR / name)

print(json.dumps(manifest, indent=2, default=float))

                          method  cv_mean_bacc  cv_std_bacc  full_val_bacc  \
0  NoTTA_StackingBlend_ClassBias      0.821134     0.028126       0.820332   
1      OptimizedWeightedEnsemble      0.818584     0.024204       0.817714   
2            NoTTA_StackingBlend      0.817028     0.024193       0.816158   

   fold_wins_over_no_tta  macro_f1  full_val_gain_over_no_tta  
0                      4  0.781634                   0.032485  
1                      5  0.804625                   0.029867  
2                      4  0.777824                   0.028312  

                            model       category         split  n_samples  \
0             NoTTA_StackingBlend       Ensemble  InternalTest       2534   
1   NoTTA_StackingBlend_ClassBias  ProposedFinal  InternalTest       2534   
2                    SimCLR_NoTTA         Single  InternalTest       2534   
3       OptimizedWeightedEnsemble       Ensemble  InternalTest       2534   
4   PreviousFixedWeightedEnsemble       Ensemble  InternalTest       2534   
5              SimCLR_SelectedTTA         Single  InternalTest       2534   
6                  ResidualFusion         Fusion  InternalTest       2534   
7                    FusionConcat         Fusion  InternalTest       2534   
8                   FusionGatedV3         Fusion  InternalTest       2534   
9                   FusionGatedV2         Fusion  InternalTest       2534   
10                       Rotation         Single  InternalTest       2534   

    accuracy      bacc  macro_precision  macro_recall  macro_f1  weighted_f

{
  "protocol_name": "Original thesis stratified 80/10/10 internal protocol",
  "status": "LOCKED",
  "primary_internal_protocol": true,
  "dataset": "ISIC 2019",
  "num_classes": 8,
  "class_names": [
    "MEL",
    "NV",
    "BCC",
    "AK",
    "BKL",
    "DF",
    "VASC",
    "SCC"
  ],
  "seed": 42,
  "split_method": "sklearn train_test_split: first 80/20, then 20% remainder split 50/50; shuffle=True; stratify=label; random_state=42",
  "split_sizes": {
    "train": 20264,
    "validation": 2533,
    "test": 2534
  },
  "final_method": "NoTTA_StackingBlend_ClassBias",
  "final_internal_test_metrics": {
    "method": "NoTTA_StackingBlend_ClassBias",
    "split": "InternalTest",
    "n_samples": 2534,
    "accuracy": 0.8389897395422258,
    "bacc": 0.8395491150909984,
    "macro_precision": 0.7666518267456637,
    "macro_recall": 0.8395491150909984,
    "macro_f1": 0.7979609543641538,
    "weighted_f1": 0.8415086081186914,
    "macro_specificity": 0.9728387154861912,
    "auc_macro_

In [9]:
# CELL 9 — CREATE COMPLETION MARKER + ZIP PACKAGE

marker = {
    "stage": "J1_original_protocol_lock",
    "status": "complete",
    "protocol": "original_stratified_80_10_10",
    "seed": 42,
    "train_size": len(TRAIN_DF),
    "validation_size": len(VAL_DF),
    "test_size": len(TEST_DF),
    "final_method": FINAL_CONFIG["final_method"],
    "final_test_bacc": float(TEST_METRICS["bacc"]),
    "test_used_for_selection": False,
    "combined_protocol_sha256": COMBINED_PROTOCOL_HASH,
    "output_rel": "j1_original_protocol_lock",
    "portable_split_rel": (
        "j1_original_protocol_lock/tables/"
        "original_80_10_10_locked_ids.csv"
    ),
}

MARKER_PATH = WORKING / "J1_PROTOCOL_LOCK_COMPLETE.json"
with open(MARKER_PATH, "w") as f:
    json.dump(marker, f, indent=2)

shutil.copy2(
    MARKER_PATH,
    OUTPUT_DIR / "J1_PROTOCOL_LOCK_COMPLETE.json",
)

ZIP_PATH = WORKING / "J1_ORIGINAL_PROTOCOL_LOCK_PACKAGE.zip"

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for p in sorted(OUTPUT_DIR.rglob("*")):
        if p.is_file():
            zf.write(
                p,
                arcname=str(p.relative_to(OUTPUT_DIR.parent))
            )

required = [
    OUTPUT_DIR / "J1_ORIGINAL_PROTOCOL_MANIFEST.json",
    TABLE_DIR / "original_80_10_10_locked_ids.csv",
    TABLE_DIR / "final_model_recomputed_metrics.csv",
    AUDIT_DIR / "protocol_hashes.json",
    AUDIT_DIR / "final_model_selection_audit.json",
    MARKER_PATH,
    ZIP_PATH,
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing expected J1 outputs:\n" + "\n".join(missing)
    )

print("=" * 92)
print("J1 ORIGINAL PROTOCOL LOCK COMPLETE")
print("=" * 92)
print("TRAIN / VAL / TEST:", len(TRAIN_DF), len(VAL_DF), len(TEST_DF))
print("Final method:", FINAL_CONFIG["final_method"])
print("Final internal BACC:", f"{100 * TEST_METRICS['bacc']:.4f}%")
print("Protocol SHA-256:", COMBINED_PROTOCOL_HASH)
print("ZIP package:", ZIP_PATH)
print()
print("Save this Kaggle notebook Version WITH OUTPUTS.")
print("Use the saved J1 output as input for J2 (DINOv2 original split).")

J1 ORIGINAL PROTOCOL LOCK COMPLETE
TRAIN / VAL / TEST: 20264 2533 2534
Final method: NoTTA_StackingBlend_ClassBias
Final internal BACC: 83.9549%
Protocol SHA-256: b483ced043583491ed0793a5e80ecd1192055230393b6603200a3441d04f5766
ZIP package: /kaggle/working/J1_ORIGINAL_PROTOCOL_LOCK_PACKAGE.zip

Save this Kaggle notebook Version WITH OUTPUTS.
Use the saved J1 output as input for J2 (DINOv2 original split).
